In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

DATA_DIR = Path("data")

META_PATH = DATA_DIR / "rag_chunks_metadata.parquet"
EMB_PATH = DATA_DIR / "rag_chunk_embeddings.npy"

META_PATH, EMB_PATH


(PosixPath('data/rag_chunks_metadata.parquet'),
 PosixPath('data/rag_chunk_embeddings.npy'))

In [10]:
meta_df = pd.read_parquet(META_PATH)
embeddings = np.load(EMB_PATH)

print("Meta shape:", meta_df.shape)
print("Embeddings shape:", embeddings.shape)
meta_df.head()


Meta shape: (2173, 6)
Embeddings shape: (2173, 768)


,chunk_id,article_id,source,title,chunk_index,chunk_text
0,1_0,1,Al Jazeera,What is the political agenda of artificial int...,0,Could AI single-handedly decide the course of ...
1,1_1,1,Al Jazeera,What is the political agenda of artificial int...,1,"of hyperrealistic, AI-generated content, such ..."
2,1_2,1,Al Jazeera,What is the political agenda of artificial int...,2,all era-defining technology – comes with consi...
3,1_3,1,Al Jazeera,What is the political agenda of artificial int...,3,"like 2001: A Space Odyssey, Blade Runner and T..."
4,1_4,1,Al Jazeera,What is the political agenda of artificial int...,4,a clear distinction between symbolic machine ‘...


In [11]:
embed_model_name = "sentence-transformers/all-mpnet-base-v2"
embedder = SentenceTransformer(embed_model_name)
embedder


SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [12]:
nn_index = NearestNeighbors(
    n_neighbors=10,
    metric="cosine",
    algorithm="auto"
)
nn_index.fit(embeddings)

print("NearestNeighbors index built.")


NearestNeighbors index built.


In [13]:
def retrieve_chunks(query, top_k=5):
    """
    Given a natural-language query, return top_k most similar chunks
    with their distances (smaller distance = more similar).
    """
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = nn_index.kneighbors(q_emb, n_neighbors=top_k)

    idxs = indices[0]
    dists = distances[0]

    results = meta_df.iloc[idxs].copy()
    results["distance"] = dists
    return results


In [14]:
retrieve_chunks("What are the risks of AI in warfare?", top_k=3)[
    ["article_id", "source", "title", "chunk_index", "distance"]
]


,article_id,source,title,chunk_index,distance
82,22,Al Jazeera,Project Force: AI and the military – a friend ...,0,0.291212
90,22,Al Jazeera,Project Force: AI and the military – a friend ...,8,0.299593
88,22,Al Jazeera,Project Force: AI and the military – a friend ...,6,0.312422


In [15]:
gen_model_name = "google/flan-t5-base"  # or "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name)

gen_model.eval()


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [16]:
def generate_text(prompt, max_new_tokens=160):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # deterministic
            num_beams=4        # small beam search for quality
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text.strip()


In [17]:
def baseline_answer(question):
    prompt = (
        "You are answering questions about news articles using only your general knowledge.\n\n"
        f"Question: {question}\n"
        "Answer in 2–4 sentences:\n"
    )
    return generate_text(prompt, max_new_tokens=180)


In [18]:
q = "What are the main concerns about AI surveillance?"
print(baseline_answer(q))


Artificial Intelligence


In [19]:
def rag_answer(question, top_k=5):
    # 1) Retrieve relevant chunks
    retrieved = retrieve_chunks(question, top_k=top_k)

    # 2) Build context with inline citations
    context_parts = []
    citations = []
    for _, row in retrieved.iterrows():
        cite = (
            f"[article_id={row.article_id}, "
            f"source={row.source}, "
            f"title={row.title}, "
            f"chunk_index={row.chunk_index}]"
        )
        citations.append(cite)
        context_parts.append(f"{cite}\n{row.chunk_text}")

    context = "\n\n".join(context_parts)

    # 3) Build Flan-T5 prompt
    prompt = (
        "You are an assistant answering questions about AI-related news.\n"
        "ONLY use the information in the context below. "
        "If the answer is not clearly supported by the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer in 2–4 sentences based ONLY on the context:\n"
    )

    answer = generate_text(prompt, max_new_tokens=220)
    return answer, citations


In [22]:
q = "When did the EU announce new AI regulations?"
ans, cits = rag_answer(q, top_k=5)

print("RAG answer:\n", ans)
print("\nCitations used:")
for c in cits:
    print(" -", c)


RAG answer:
 EU watchdogs have said.

Citations used:
 - [article_id=9, source=Al Jazeera, title=EU parliament greenlights landmark artificial intelligence regulations ..., chunk_index=1]
 - [article_id=9, source=Al Jazeera, title=EU parliament greenlights landmark artificial intelligence regulations ..., chunk_index=3]
 - [article_id=95, source=Al Jazeera, title=European Union reaches agreement on landmark legislation to ..., chunk_index=0]
 - [article_id=86, source=Al Jazeera, title='Setting the standard': EU unveils plan to rein in risky AI uses ..., chunk_index=2]
 - [article_id=26, source=Al Jazeera, title=Governments race to regulate artificial intelligence tools ..., chunk_index=1]
